In [1]:
path = '/kaggle/working/'

import numpy as np
import pandas as pd

import ast,shutil,copy
import warnings
from bokeh.plotting import figure, gridplot 
from bokeh.io import output_file, show, output_notebook; output_notebook()

warnings.filterwarnings('ignore')

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


Loading BokehJS ...

/kaggle/input/datasets/yusufmurtaza01/s6e2d8/0.95401.csv
/kaggle/input/datasets/yusufmurtaza01/s6e2d8/0.95400.csv
/kaggle/input/datasets/yusufmurtaza01/s6e2d7/0.95399.csv
/kaggle/input/datasets/yusufmurtaza01/s6e2d7/0.95397.csv
/kaggle/input/datasets/yusufmurtaza01/s6e2d7/0.95396.csv
/kaggle/input/datasets/yusufmurtaza01/s6e2d7/0.95398.csv
/kaggle/input/datasets/yusufmurtaza01/s6e2d9/0.95403.csv
/kaggle/input/datasets/yusufmurtaza01/s6e2d9/0.95405.csv
/kaggle/input/datasets/yusufmurtaza01/s6e2d9/0.95406.csv
/kaggle/input/datasets/yusufmurtaza01/s6e2d9/0.95401.csv
/kaggle/input/datasets/yusufmurtaza01/s6e2-d11/0.95406.csv
/kaggle/input/datasets/yusufmurtaza01/s6e2-d11/0.95409.csv
/kaggle/input/datasets/yusufmurtaza01/s6e2-d11/0.95408.csv
/kaggle/input/datasets/yusufmurtaza01/s6e2-d11/0.95407.csv
/kaggle/input/datasets/yusufmurtaza01/s6e2d6/0.95390.csv
/kaggle/input/datasets/yusufmurtaza01/s6e2d6/0.95387.csv
/kaggle/input/datasets/yusufmurtaza01/s6e2d6/0.95393.csv
/kaggle/input/datasets/

In [2]:
def bokeh_show(
        params,
        df_cross,
        show_figures1, 
        show_figures2, wps_fig2,
        color_cross):

    colors = [subm['color'] for subm in params['subm']]
    
    def dossier(js,subms,cols):
        def quant(i,js,subms,cols):
            return {"c" : i, "q" : sum([1 for subm in cols[i] if subm == subms[js]])}
        return {
            'name' : subms[js],
            'q_in' : [quant(i,js,subms,cols) for i in range(len(subms))]
        }
    alls = pd.read_csv(f'tida_desc.csv')
    matrix = [ast.literal_eval(str(row.alls)) for row in alls.itertuples()]
    subms = sorted(matrix[0])
    cols = [[data[i] for data in matrix] for i in range(len(subms))]
    df_subms = pd.DataFrame({f'col_{i}': [x[i] for x in matrix] for i in range(len(subms))})
    dossiers = [dossier(js,subms,cols) for js in range(len(subms))]
    subm_names = [one_dossier['name'] for one_dossier in dossiers]
    figures1,qss,i = [],[],0
    height = 100 if len(colors)==2\
        else 134 if len(colors)==3 else (154 if len(colors)==4 else 174)
    for one_dossier in dossiers: 
        i_col = 'alls. ' + str(one_dossier['q_in'][i]['c'])
        qs = [one['q'] for one in one_dossier['q_in']]
        x_names = [name.replace("Group","").replace("subm_","") for name in subm_names]
        width = 140
        f = figure(x_range=x_names,width=width, height=height, title=i_col)
        f.vbar(x=x_names, width=0.585, top=qs, color=colors)
        figures1.append(f)
        qss.append(qs)
        i+=1
    grid = gridplot([figures1])
    output_file('tida_alls.html')
    if show_figures1 == True: show(grid)
    sub_wts = params['subwts']
    main_wts = [subm['weight'] for subm in params['subm']]
    mms,acc_mass = [],[]
    for j in range(len(dossiers)):
        one_dossier = dossiers[j]
        qs = [one['q'] for one in one_dossier['q_in']]
        mm = [qs[h] * (main_wts[j] + sub_wts[h]) for h in range(len(sub_wts))]
        mass = sum(mm)
        mms.append(mm)
        acc_mass.append(round(mass))                        #subm_names[::-1]
    y_names = [name + " - " + str(mass) for name,mass in zip(subm_names,acc_mass)]
    f1 = figure(y_range=y_names, width=270, height=height, title='relations of general masses')
    f1.hbar(y=y_names, height=0.555, right=acc_mass, left=0, color=colors)
    output_file('tida_alls2.html')
    alls = [f'alls.{i}' for i in range(len(dossiers))]
    subm = [f'sub{i}'   for i in range(len(dossiers))] 
    mmsT  = np.asarray(mms).T
    data = {'cols' : alls}
    for i in range(len(dossiers)): data[f'sub{i}'] = mmsT[i,:]
    f2 = figure(y_range=alls, height=height, width=270, title="relations of columns masses")
    f2.hbar_stack(subm, y='cols', height=0.555, color=colors, source=data)
    qssT  = np.asarray(qss).T
    data = {'cols' : alls}
    for i in range(len(dossiers)): data[f'sub{i}'] = qssT[i,:]
    f3 = figure(y_range=alls, height=height, width=245, title="ratios in columns")
    f3.hbar_stack(subm, y='cols', height=0.555, color=colors, source=data)
    grid = gridplot([[f3,f2,f1]])
    show(grid)
    if show_figures2 == True:
        def read(params,i):
            FiN = params["path"] + params["subm"][i]["name"] + ".csv"
            target_name_back = {'target':params["target"],'pred':params["target"]}
            return pd.read_csv(FiN).rename(columns=target_name_back)
        dfs = [read(params,i) for i in range(len(params["subm"]))] + [df_cross]
        _height = 358 if len(params["subm"]) == 11 else 254
        f   = figure(width=785, height=_height)
        f.title.text = 'Click on legend entries to mute the corresponding lines'
        b,e        = 21000,21154
        line_x     = [dfs[i][b:e]['id']         for i in range(len(dfs))]
        line_y     = [dfs[i][b:e]['exam_score'] for i in range(len(dfs))]
        color      = colors + [color_cross]
        alpha      = [0.8 for i in range(len(dfs)-1)] + [0.95]
        lws        = [1.0 for i in range(len(dfs)-1)] + [1.00]
        legend = subm_names + ['cross']
        for i in range(len(legend)):
            f.line(line_x[i], line_y[i], line_width=lws[i], color=color[i], alpha=alpha[i],
                   muted_color='white',legend_label=legend[i])
        f.legend.location = "top_left"
        f.legend.click_policy="mute"
        show(f)

In [3]:
def matrix_vs(path,fs_names):
    def load(path,fs_names):
        dfs = [pd.read_csv(path + name_subm +'.csv') for name_subm in fs_names]
        for i in range(len(dfs)):
            dfs[i] = dfs[i].rename(columns={"exam_score": f'{fs_names[i]}'})
        dfsm = pd.merge(dfs[0], dfs[1], on="id")
        for i in range(2,len(dfs)):
            dfsm = pd.merge(dfsm,dfs[i],on='id')
        return dfsm   
    def make_list_vs(fs_names):
        list = []
        for i in range(0,len(fs_names)-1):
            for j in range(i+1,len(fs_names)):
                list.append(fs_names[i] + "_vs_" + fs_names[j])
        return list
    def get_mvs(dfs, list_vs):
        def get_abs_distance(x,t1,t2):
            return abs(x[t1]-x[t2])
        for vs in list_vs:
            t = vs.split('_vs_')
            dfs[vs] = dfs.apply(lambda x: get_abs_distance(x,t[0],t[1]), axis=1)
        return dfs   
    def distance_vs(name, st_names, list_vs, dfs):
        distances = []
        for st in st_names:
            vs_between = name + "_vs_" + st
            if vs_between not in list_vs:
                distances.append(0)
            else: distances.append(round(dfs[vs_between].sum()))
        return distances
    dfs = load(path,fs_names)
    list_vs = make_list_vs(fs_names)
    mvs = get_mvs(dfs, list_vs)
    m1 = pd.DataFrame({'subm':fs_names})
    m2 = pd.DataFrame({ name :distance_vs(name, fs_names, list_vs, mvs) for name in fs_names})
    matrix = pd.concat([m1,m2],axis=1)
    return matrix


def seaborn_Show(params,file_name_cross=''):
    import matplotlib.pyplot as plt, seaborn as sns
    import warnings; warnings.filterwarnings('ignore')
    plt.figure(figsize=(8.7, 2))
    for subm in params['subm']:
        pred = pd.read_csv(params['path']+subm['name']+'.csv')[params['id_target'][1]]
        sns.kdeplot(pred, label = subm['name'], linewidth = 0.5)
    if file_name_cross != '':
        pred = pd.read_csv(file_name_cross)[params['id_target'][1]]
        sns.kdeplot(pred, label = 'blend', linewidth = 1, linestyle = 'dashed')
    plt.title("KDE")
    plt.xlabel("target")
    plt.ylabel("Density")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


def display_distances(params):
    files = [subm['name'] for subm in params['subm']]
    distances = matrix_vs ( params['path'], files )            
    display(distances)


def arr_colors(color):
    dskb,mvr = 'deepskyblue','mediumvioletred'
    sg = ['darkgray','silver','gainsboro']
    if color=='red'   or color=='R': return ['firebrick','red','crimson','tomato']     + sg
    if color=='Red'   or color=='r': return ['red','tomato','crimson']                 + sg
    if color=='Green' or color=='G': return ['darkgreen','limegreen','green','lime']   + sg
    if color=='Blue'  or color=='B': return ['midnightblue','blue','mediumblue',dskb]  + sg
    if color=='RGB'   or color=='S': return ['mediumblue','darkgreen','crimson']       + sg
    if color=='RGBM'  or color=='M': return [mvr,'darkorchid','darkmagenta','magenta'] + sg
    return ['black','dimgray','gray'] + sg


def convert(schema):
    colors = arr_colors(schema[2])
    dicts  = [
        {'name': schema[0][i],'weight':schema[1][i],'color':colors[i]} 
        for i in range(len(schema[0]))
    ]
    return {'subm':dicts}

In [4]:
def h_blend(
        params, _update={},
        cross='silver',
        details=True,
        fig1=False, fig2=False, wf2=555, 
        dtls=False, dist=False, subm=''):

    if isinstance(params, list): params = convert(params)

    if 'path' in _update or 'subwts' in _update : params.update(_update)
    
    color_cross, dk  = cross, copy.deepcopy(params)

    if details == True:
        dist = True
        show_details,show_figures1,show_figures2 = True,True,True
    else:
        show_details,show_figures1,show_figures2 = dtls,fig1,fig2
        
    file_short_names = [subm['name'] for subm in params['subm']]
    type_sort    = params['type_sort'][0]
    dk['asc']    = params['type_sort'][1]
    dk['desc']   = params['type_sort'][2]
    dk['id']     = params['id_target'][0]
    dk['target'] = params['id_target'][1]

    def read(dk,i):
        tnm = dk["subm"][i]["name"]
        FiN = dk["path"] + tnm + ".csv"
        return pd.read_csv(FiN).rename(columns={
            'target':tnm, 'pred':tnm, dk["target"]:tnm})
        
    def merge(dfs_subm):
        df_subms = pd.merge(dfs_subm[0],  dfs_subm[1], on=[dk['id']])
        for i in range(2, len(dk["subm"])): 
            df_subms = pd.merge(df_subms, dfs_subm[i], on=[dk['id']])
        return df_subms
        
    def da(dk,sorting_direction,show_details):
        
        df_subms = merge([read(dk,i) for i in range(len(dk["subm"]))])
        cols = [col for col in df_subms.columns if col != dk['id']]
        short_name_cols = [c for c in cols]
        
        def alls1(x, sd=sorting_direction,cs=cols):
            reverse = True if sd=='desc' else False
            tes = {c: x[c] for c in cs}.items()
            subms_sorted = [t[0] for t in sorted(tes,key=lambda k:k[1],reverse=reverse)]
            return subms_sorted

        import random

        def alls2(x, sd=sorting_direction,cs=cols):
            reverse = True if sd=='desc' else False
            tes = {c: x[c] for c in cs}.items()
            subms_random = [t[0] for t in tes]
            random.shuffle(subms_random)
            return subms_random

        alls = alls1 if type_sort == 'asc/desc' else alls2
            
        def summa(x,cs,wts,ic_alls): 
            return sum([x[cs[j]] * (wts[0][j] + wts[1][ic_alls[j]]) for j in range(len(cs))])
            
        wts = [[[e['weight'] for e in dk["subm"]], [w for w in dk["subwts"]]]]
          
        def correct(x, cs=cols, wts=wts):
            i = [x['alls'].index(c) for c in short_name_cols]
            return summa(x,cs,wts[0],i)

        if len(wts) == 1:
            correct_sub_weights = [wt for wt in dk["subwts"]]
            weights = [subm['weight'] for subm in dk["subm"]]
            def correct(x, cs=cols, w=weights, cw=correct_sub_weights):
                ic = [x['alls'].index(c) for c in short_name_cols]
                cS = [x[cols[j]] * (w[j] + cw[ic[j]]) for j in range(len(cols))]
                return sum(cS)
                
        if len(wts) > 1 or "subwts2" in dk:

            wts = [
                [[e['weight'] for e in dk["subm"]], [w for w in dk["subwts" ]]],
                [[e['weight'] for e in dk["subm2"]],[w for w in dk["subwts2"]]],
                [[e['weight'] for e in dk["subm3"]],[w for w in dk["subwts3"]]],
            ]

            def correct(x, cs=cols, wts=wts):
                i = [x['alls'].index(c) for c in short_name_cols]
                if   0.0540 < x['mx-m'] <= 0.0740: return summa(x,cs,wts[2],i)
                if   0.0000 < x['mx-m'] <= 0.0050: return summa(x,cs,wts[1],i)
                else:                              return summa(x,cs,wts[0],i)
                   
        def amxm(x, cs=cols):
            list_values = x[cs].to_list()
            mxm = abs(max(list_values)-min(list_values))
            return mxm

        if len(wts) > 1 or "subwts2" in dk:
            df_subms['mx-m']   = df_subms.apply(lambda x: amxm   (x), axis=1)
        df_subms['alls']       = df_subms.apply(lambda x: alls   (x), axis=1)
        df_subms[dk["target"]] = df_subms.apply(lambda x: correct(x), axis=1)
        schema_rename = { old_nc:new_shnc for old_nc, new_shnc in zip(cols, short_name_cols) }
        df_subms = df_subms.rename(columns=schema_rename)
        df_subms = df_subms.rename(columns={dk["target"]:"ensemble"})
        df_subms.insert(loc=1, column=' _ ', value=['   '] * len(df_subms))
        df_subms[' _ '] = df_subms[' _ '].astype(str)
        pd.set_option('display.max_rows',100)
        pd.set_option('display.float_format', '{:.5f}'.format)
        if len(wts) > 1: 
            vcols = [dk['id']] + [' _ '] + short_name_cols + [' _ '] + ['mx-m'] + [' _ '] +\
                      ['alls'] + [' _ '] + ['ensemble']
        else:
            vcols = [dk['id']] + [' _ '] + short_name_cols + [' _ '] +\
                      ['alls'] + [' _ '] + ['ensemble']
        df_subms = df_subms[vcols]
        if show_details and sorting_direction=='desc': display(df_subms.head(5))
        pd.set_option('display.float_format', '{:.5f}'.format)
        df_subms = df_subms.rename(columns={"ensemble":dk["target"]})
        if sorting_direction=='desc': 
            df_subms.to_csv(f'tida_{sorting_direction}.csv', index=False)
        return df_subms[[dk['id'],dk['target']]]
   
    def ensemble_da(dk,        show_details): 
        dfD    = da(dk,'desc', show_details)
        dfA    = da(dk,'asc',  show_details)
        dfA[dk['target']] = dk['desc']*dfD[dk['target']] + dfA[dk['target']]*dk['asc']
        return dfA

    da = ensemble_da(dk,show_details)

    if subm != '': da.to_csv(subm, index=False)
        
    return  da

In [5]:
def shutil_copy(inst,dest):
    shutil.copy(inst, dest)
    return pd.read_csv(dest)


def b2(fin0,fin1,wts,out):
    df = pd.read_csv('/kaggle/input/playground-series-s6e2/sample_submission.csv')
    df0 = pd.read_csv(path + fin0 + '.csv')
    df1 = pd.read_csv(path + fin1 + '.csv')
    df['Heart Disease'] = \
        df0['Heart Disease'] * wts[0] + df1['Heart Disease'] * wts[1]
    df.to_csv(out,index=False)                  
    return df


# Ensure these paths in your b2 function match your input list
def b2(df0, wts, df1, subm='submission.csv'):
    df = pd.read_csv('/kaggle/input/playground-series-s6e2/sample_submission.csv')
    df['Heart Disease'] = df0['Heart Disease'] * wts[1] + df1['Heart Disease'] * wts[0]
    df.to_csv(subm, index=False)


def para(df_Main, weights, dfs_Aux, params_Aux):
    for i in range(len(dfs_Aux)):
        b2(df_Main, weights, dfs_Aux[i], params_Aux['subm'][i]['name']+'.csv')
    return copy.deepcopy(params_Aux)


In [6]:
weights1,weights2,weights3 = [0.96,0.04], [0.89,0.11], [0.82,0.18]

ct1,ct2 = 1.00118, 1.00118

# Custom Main files

In [7]:
params_Main = {
      'path'     : f'/kaggle/input/datasets/yusufmurtaza01/s6e2-d11/', 
      'id_target': ['id',"Heart Disease"],
      'type_sort': ['asc/desc', 0.30, 0.70],
      'subm'     : [
          {'name': '0.95409', 'weight': +0.31, 'color': 'darkmagenta'}, # Best file
          {'name': '0.95408', 'weight': +0.33, 'color': 'darkgreen'},
          {'name': '0.95407', 'weight': +0.17, 'color': 'deeppink'},
          {'name': '0.95406', 'weight': +0.19, 'color': 'magenta'},
      ]
}

In [8]:
params_Aux = {
      'path'     : f'/kaggle/working/',
      'id_target': ['id',"Heart Disease"],        
      'type_sort': ['asc/desc', 0.30, 0.70],
      'subwts'   : [-0.25, 0.00, +0.55, -0.30],
      'subm'     : [
          {'name': 'Main+24', 'weight': +0.21, 'color': 'brown'},
          {'name': 'Main+25', 'weight': +0.08, 'color': 'navy' },
          {'name': 'Main+28', 'weight': +0.23, 'color': 'black'},
          {'name': 'Main+29', 'weight': +0.48, 'color': 'red'  },
      ]
}


In [9]:
input_path = '/kaggle/input/datasets/yusufmurtaza01/s6e2d9/'

df24 = pd.read_csv(input_path + '0.95405.csv') # High quality secondary
df25 = pd.read_csv(input_path + '0.95403.csv') # High quality secondary
df28 = pd.read_csv(input_path + '0.95401.csv') # Diversity file
df29 = pd.read_csv(input_path + '0.95406.csv') # Baseline file

dfs_Aux = [df24, df25, df28, df29]
dfs_Aux = [df24, df25, df28, df29]

In [10]:
df_Main = h_blend(params_Main,_update={'subwts':[+0.55, -0.10, -0.20, -0.25]})

df1 = h_blend( para(df_Main, weights1, dfs_Aux, params_Aux) )

df_Main = h_blend(params_Main,_update={'subwts':[+0.11, -0.01, -0.03, -0.07]})

df2 = h_blend( para(df_Main, weights2, dfs_Aux, params_Aux) )

df_Main = h_blend(params_Main,_update={'subwts':[+0.55, -0.10, -0.20, -0.25]})

df3 = h_blend( para(df_Main, weights3, dfs_Aux, params_Aux) )

,id,_,0.95409,0.95408,0.95407,0.95406,_,alls,_,ensemble
0,630000,,0.94814,0.92910,0.90957,0.94990,,"[0.95406, 0.95409, 0.95408, 0.95407]",,0.95005
1,630001,,0.01230,0.01174,0.02061,0.01170,,"[0.95407, 0.95409, 0.95408, 0.95406]",,0.01825
2,630002,,0.98729,0.97408,0.97039,0.98810,,"[0.95406, 0.95409, 0.95408, 0.95407]",,0.98752
3,630003,,0.00761,0.00685,0.00891,0.00740,,"[0.95407, 0.95409, 0.95406, 0.95408]",,0.00849
4,630004,,0.22299,0.21516,0.25182,0.22040,,"[0.95407, 0.95409, 0.95406, 0.95408]",,0.24315


,id,_,Main+24,Main+25,Main+28,Main+29,_,alls,_,ensemble
0,630000,,0.94902,0.92511,0.94892,0.94950,,"[Main+29, Main+24, Main+28, Main+25]",,0.95431
1,630001,,0.01140,0.02040,0.01106,0.01188,,"[Main+25, Main+29, Main+24, Main+28]",,0.01012
2,630002,,0.98799,0.97619,0.98800,0.98789,,"[Main+28, Main+24, Main+29, Main+25]",,0.99048
3,630003,,0.00723,0.01055,0.00654,0.00742,,"[Main+25, Main+29, Main+24, Main+28]",,0.00681
4,630004,,0.22230,0.25550,0.22307,0.22095,,"[Main+25, Main+28, Main+24, Main+29]",,0.21659


,id,_,0.95409,0.95408,0.95407,0.95406,_,alls,_,ensemble
0,630000,,0.94814,0.92910,0.90957,0.94990,,"[0.95406, 0.95409, 0.95408, 0.95407]",,0.93910
1,630001,,0.01230,0.01174,0.02061,0.01170,,"[0.95407, 0.95409, 0.95408, 0.95406]",,0.01439
2,630002,,0.98729,0.97408,0.97039,0.98810,,"[0.95406, 0.95409, 0.95408, 0.95407]",,0.98188
3,630003,,0.00761,0.00685,0.00891,0.00740,,"[0.95407, 0.95409, 0.95406, 0.95408]",,0.00774
4,630004,,0.22299,0.21516,0.25182,0.22040,,"[0.95407, 0.95409, 0.95406, 0.95408]",,0.22861


,id,_,Main+24,Main+25,Main+28,Main+29,_,alls,_,ensemble
0,630000,,0.94801,0.92585,0.94792,0.94846,,"[Main+29, Main+24, Main+28, Main+25]",,0.95292
1,630001,,0.01150,0.01984,0.01119,0.01194,,"[Main+25, Main+29, Main+24, Main+28]",,0.01031
2,630002,,0.98739,0.97645,0.98740,0.98730,,"[Main+28, Main+24, Main+29, Main+25]",,0.98971
3,630003,,0.00725,0.01033,0.00661,0.00743,,"[Main+25, Main+29, Main+24, Main+28]",,0.00685
4,630004,,0.22233,0.25311,0.22305,0.22108,,"[Main+25, Main+28, Main+24, Main+29]",,0.21704


,id,_,0.95409,0.95408,0.95407,0.95406,_,alls,_,ensemble
0,630000,,0.94814,0.92910,0.90957,0.94990,,"[0.95406, 0.95409, 0.95408, 0.95407]",,0.95005
1,630001,,0.01230,0.01174,0.02061,0.01170,,"[0.95407, 0.95409, 0.95408, 0.95406]",,0.01825
2,630002,,0.98729,0.97408,0.97039,0.98810,,"[0.95406, 0.95409, 0.95408, 0.95407]",,0.98752
3,630003,,0.00761,0.00685,0.00891,0.00740,,"[0.95407, 0.95409, 0.95406, 0.95408]",,0.00849
4,630004,,0.22299,0.21516,0.25182,0.22040,,"[0.95407, 0.95409, 0.95406, 0.95408]",,0.24315


,id,_,Main+24,Main+25,Main+28,Main+29,_,alls,_,ensemble
0,630000,,0.94767,0.92725,0.94759,0.94808,,"[Main+29, Main+24, Main+28, Main+25]",,0.95219
1,630001,,0.01208,0.01977,0.01179,0.01249,,"[Main+25, Main+29, Main+24, Main+28]",,0.01099
2,630002,,0.98724,0.97716,0.98725,0.98716,,"[Main+28, Main+24, Main+29, Main+25]",,0.98937
3,630003,,0.00734,0.01018,0.00675,0.00751,,"[Main+25, Main+29, Main+24, Main+28]",,0.00698
4,630004,,0.22404,0.25241,0.22470,0.22290,,"[Main+25, Main+28, Main+24, Main+29]",,0.21917


In [11]:
df1.rename(columns={'Heart Disease':'es1'},inplace=True)
df2.rename(columns={'Heart Disease':'es2'},inplace=True)
df3.rename(columns={'Heart Disease':'es3'},inplace=True)

df = pd.merge(df1,df2,on='id')
df = pd.merge(df, df3,on='id')


def trend(x):
    e1,e2,e3 = x['es1'],x['es2'],x['es3']
    if e1 < e3 and e2 < e3: return x['es3'] * (ct1 - 0.0001*(e3-e1))  
    if e1 > e2 and e2 > e3: return x['es3'] / (ct2 - 0.0001*(e1-e3))
    return x['es3']


df['Heart Disease'] = df.apply(lambda x: trend(x), axis=1)

df

,id,es1,es2,es3,Heart Disease
0,630000,0.95396,0.95260,0.95190,0.95077
1,630001,0.01006,0.01026,0.01094,0.01095
2,630002,0.99032,0.98956,0.98924,0.98807
3,630003,0.00678,0.00683,0.00695,0.00696
4,630004,0.21620,0.21668,0.21883,0.21909
...,...,...,...,...,...
269995,899995,0.15538,0.15602,0.15863,0.15881
269996,899996,0.69020,0.68965,0.68958,0.68877
269997,899997,0.04569,0.04636,0.04882,0.04888
269998,899998,0.17810,0.17874,0.18127,0.18148


In [12]:
for name in '4,5,8,9'.split(','):
    file = f'/kaggle/working/Main+2{name}.csv'
    if os.path.isfile(file): os.remove(file)

In [13]:
df[['id','Heart Disease']].to_csv('submission_bokeh.csv',index=False)
df[['id','Heart Disease']]

,id,Heart Disease
0,630000,0.95077
1,630001,0.01095
2,630002,0.98807
3,630003,0.00696
4,630004,0.21909
...,...,...
269995,899995,0.15881
269996,899996,0.68877
269997,899997,0.04888
269998,899998,0.18148


# Top 4

In [14]:
import os
import re

# 1. Set Path
INPUT_PATH = '/kaggle/input/datasets/yusufmurtaza01/s6e2-d11/'

# 2. Robust Detection & Sorting
all_files = [f for f in os.listdir(INPUT_PATH) if f.endswith('.csv') and 'noscore' not in f]

def get_numeric_score(filename):
    # This regex finds the first sequence of numbers and dots (e.g., 8.54466)
    # even if there are suffixes like .ct or .ct.csv
    match = re.search(r"(\d+\.\d+)", filename)
    if match:
        return float(match.group(1))
    return 99.9  # Fallback for files without a clear score

# Sort by the extracted numeric score (lowest error first)
all_files.sort(key=get_numeric_score, reverse=True)
# Take the top 7
# found_files = all_files[:7]
# print(f"Sorted Top 7 files: {found_files}")

# # Take only the top 4
found_files = all_files[:4]
# found_files.append("8.70199.csv")
print(f"Sorted Top 4 files: {found_files}")

Sorted Top 4 files: ['0.95409.csv', '0.95408.csv', '0.95407.csv', '0.95406.csv']


In [15]:
#  Create Submission List
subm_list = []
# Defined a larger color palette for 7 files
# colors = ['#FF4136', '#B10DC9', '#2ECC40', '#0074D9', '#FF851B', '#39CCCC', '#FFDC00']

colors = ['#FF4136', '#B10DC9', '#2ECC40', '#0074D9']
for i, filename in enumerate(found_files):
    name_no_ext = filename.replace('.csv', '')
    subm_list.append({
        'name': name_no_ext, 
        'weight': 1.0 / len(found_files), # Base weight (equal distribution)
        'color': colors[i]
    })

#  Optimized Sub-Weights for 7 files
# base_subwts = [20, 10, 5, 0, -5, -10, -20] 

base_subwts = [11, -1,-3,-7] #for top4


params = {
    'path': INPUT_PATH,
    'id_target': ['id', 'Heart Disease'], 
    'type_sort': ['asc/desc', 0.30, 0.70],     
    'subwts': [w/200 for w in base_subwts], # Scaling sub-weights
    'subm': subm_list
}

# 5. Execute the Blend
df_blend = h_blend(
    params, 
    details=True, 
    subm='submission_top4.csv' 
)

print("\nBest file used:", found_files[0])
print("Blended submission ready for download.")

,id,_,0.95409,0.95408,0.95407,0.95406,_,alls,_,ensemble
0,630000,,0.94814,0.92910,0.90957,0.94990,,"[0.95406, 0.95409, 0.95408, 0.95407]",,0.93591
1,630001,,0.01230,0.01174,0.02061,0.01170,,"[0.95407, 0.95409, 0.95408, 0.95406]",,0.01457
2,630002,,0.98729,0.97408,0.97039,0.98810,,"[0.95406, 0.95409, 0.95408, 0.95407]",,0.98080
3,630003,,0.00761,0.00685,0.00891,0.00740,,"[0.95407, 0.95409, 0.95406, 0.95408]",,0.00779
4,630004,,0.22299,0.21516,0.25182,0.22040,,"[0.95407, 0.95409, 0.95406, 0.95408]",,0.22949



Best file used: 0.95409.csv
Blended submission ready for download.


# Simple blendd of top 2

In [16]:
import pandas as pd

# Define the blending function
def blend_submissions(weight_dict, output_path):
    # Initialize list to store loaded DataFrames
    dataframes = []

    # Load each submission with its weight
    for path, weight in weight_dict.items():
        # Read the CSV file
        df = pd.read_csv(path)

        # Add a weighted prediction column
        df["weighted_pred"] = df["Heart Disease"] * weight

        # Append to list
        dataframes.append(df[["id", "weighted_pred"]])

    # Merge all submissions on 'id'
    merged = dataframes[0]
    for df in dataframes[1:]:
        # Merge on id
        merged = merged.merge(df, on="id", how="inner", suffixes=("", "_dup"))

        # Combine duplicate weighted_pred columns if any
        if "weighted_pred_dup" in merged.columns:
            merged["weighted_pred"] += merged["weighted_pred_dup"]
            merged.drop(columns=["weighted_pred_dup"], inplace=True)

    # Compute total weight
    total_weight = sum(weight_dict.values())

    # Compute blended prediction
    merged["Heart Disease"] = merged["weighted_pred"] / total_weight

    # Prepare final DataFrame
    blended = merged[["id", "Heart Disease"]]

    # Save blended submission
    blended.to_csv(output_path, index=False)

    # Print confirmation
    print(f"✅ Blended submission saved to {output_path}")
# Define the main function
def main():
    # Define file paths and their respective weights
    weight_dict = {
        "/kaggle/input/notebooks/yusufmurtaza01/s6e2-blend3/submission.csv": 2.8,
        "/kaggle/input/notebooks/yusufmurtaza01/s6e2-blend3/submission_t2.csv": 0.2,
    }
    # Call blend function
    blend_submissions(weight_dict, output_path="submission_t2.csv")

# Call the main function
if __name__ == "__main__":
    main()

✅ Blended submission saved to submission_t2.csv
